In [2]:
!hf auth login


    _|    _|  _|    _|    _|_|_|    _|_|_|  _|_|_|  _|      _|    _|_|_|      _|_|_|_|    _|_|      _|_|_|  _|_|_|_|
    _|    _|  _|    _|  _|        _|          _|    _|_|    _|  _|            _|        _|    _|  _|        _|
    _|_|_|_|  _|    _|  _|  _|_|  _|  _|_|    _|    _|  _|  _|  _|  _|_|      _|_|_|    _|_|_|_|  _|        _|_|_|
    _|    _|  _|    _|  _|    _|  _|    _|    _|    _|    _|_|  _|    _|      _|        _|    _|  _|        _|
    _|    _|    _|_|      _|_|_|    _|_|_|  _|_|_|  _|      _|    _|_|_|      _|        _|    _|    _|_|_|  _|_|_|_|

    To log in, `huggingface_hub` requires a token generated from https://huggingface.co/settings/tokens .
Enter your token (input will not be visible): ^C


In [1]:
from datasets import load_dataset
tinystories_dataset = load_dataset("roneneldan/TinyStories", split="train")
gsm8k_dataset = load_dataset("gsm8k", "main", split="train")

c:\Users\MSI-1\AppData\Local\Programs\Python\Python310\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
tinystories_dataset.shape

(2119719, 1)

In [3]:
gsm8k_dataset.shape


(7473, 2)

In [4]:
tinystories_dataset[0]

{'text': 'One day, a little girl named Lily found a needle in her room. She knew it was difficult to play with it because it was sharp. Lily wanted to share the needle with her mom, so she could sew a button on her shirt.\n\nLily went to her mom and said, "Mom, I found this needle. Can you share it with me and sew my shirt?" Her mom smiled and said, "Yes, Lily, we can share the needle and fix your shirt."\n\nTogether, they shared the needle and sewed the button on Lily\'s shirt. It was not difficult for them because they were sharing and helping each other. After they finished, Lily thanked her mom for sharing the needle and fixing her shirt. They both felt happy because they had shared and worked together.'}

In [5]:
gsm8k_dataset[0]

{'question': 'Natalia sold clips to 48 of her friends in April, and then she sold half as many clips in May. How many clips did Natalia sell altogether in April and May?',
 'answer': 'Natalia sold 48/2 = <<48/2=24>>24 clips in May.\nNatalia sold 48+24 = <<48+24=72>>72 clips altogether in April and May.\n#### 72'}

In [6]:
def get_training_corpus(tinystories_dataset, gsm8k_dataset):
    for i, row in enumerate(tinystories_dataset):
        yield row['text']
    for row in gsm8k_dataset:
        question = row['question']
        raw_answer = row['answer']

        parts = raw_answer.split('####')
        reasoning = parts[0].strip()
        final_answer = parts[1].strip() if len(parts) > 1 else ""

        formatted_text = (
            f"[INST] {question} [/INST]\n"
            f"[THINK]\n{reasoning}\n[/THINK]\n"
            f"[ANSWER] {final_answer} [/ANSWER]"
        )
        yield formatted_text
        


In [9]:
import os
from datasets import load_dataset
from tokenizers import Tokenizer, Regex
from tokenizers.models import BPE
from tokenizers.trainers import BpeTrainer
from tokenizers.pre_tokenizers import Sequence, Whitespace, Split
from tokenizers.processors import TemplateProcessing
from tokenizers import decoders

def train_hf_tokenizer(output_dir: str = "./trm_tokenizer"):
    print("Initializing BPE Tokenizer...")
    tokenizer = Tokenizer(BPE(unk_token="[UNK]", continuing_subword_prefix="##") )

    # The Math Hack: Split by whitespace, then isolate digits
    tokenizer.pre_tokenizer = Sequence([
        Whitespace(),
        Split(Regex(r"\d"), behavior="isolated") 
    ])

    # Structural Tokens for G-TRM
    special_tokens = [
        "[PAD]", "[UNK]", "[BOS]", "[EOS]", "[MASK]", 
        "[INST]", "[/INST]", "[THINK]", "[/THINK]", "[ANSWER]", "[/ANSWER]"
    ]

    # 4096 Vocab Size to keep embedding layer ~1M parameters
    trainer = BpeTrainer(
        vocab_size=4096, 
        special_tokens=special_tokens,
        initial_alphabet=list("abcdefghijklmnopqrstuvwxyzABCDEFGHIJKLMNOPQRSTUVWXYZ!@#$%^&*()_+-=[]{}|;':\",./<>?`~ \n"),
        show_progress=True,
        continuing_subword_prefix="##"
    )

    tokenizer.decoder = decoders.WordPiece(prefix="##")

    print("Training tokenizer from Hugging Face iterator...")
    # Train using the generator!
    tokenizer.train_from_iterator(get_training_corpus(tinystories_dataset, gsm8k_dataset), trainer=trainer)

    # Post-Processing: Auto-wrap sequences in BOS and EOS
    tokenizer.post_processor = TemplateProcessing(
        single="[BOS] $A [EOS]",
        pair="[BOS] $A [EOS] $B:1 [EOS]:1",
        special_tokens=[
            ("[BOS]", tokenizer.token_to_id("[BOS]")),
            ("[EOS]", tokenizer.token_to_id("[EOS]")),
        ],
    )

    # Save to disk
    if not os.path.exists(output_dir):
        os.makedirs(output_dir)
    
    save_path = os.path.join(output_dir, "tokenizer_mask_1.json")
    tokenizer.save(save_path)
    print(f"\nSuccess! Custom Micro-BPE tokenizer saved to {save_path}")

    return tokenizer

In [10]:
my_tokenizer = train_hf_tokenizer()

Initializing BPE Tokenizer...
Training tokenizer from Hugging Face iterator...






Success! Custom Micro-BPE tokenizer saved to ./trm_tokenizer/tokenizer_mask_1.json


Preparing Dataset

In [3]:
import torch
from datasets import load_dataset
from tokenizers import Tokenizer
import re

# ==========================================
# 1. Configuration & Tokenizer Setup
# ==========================================
MAX_SEQ_LEN = 512 
# DataLoader micro-batch; use 8–16 on 4 GB GPUs (64 × 512 activations OOMs easily during training)
BATCH_SIZE = 16

print("Loading Tokenizer...")
tokenizer = Tokenizer.from_file("C:/Users/MSI-1/Desktop/adnan_final/trm_tokenizer/tokenizer_mask_1.json")

# Extract special token IDs (assuming your tokenizer has these defined)
MASK_TOKEN_ID = tokenizer.token_to_id("[MASK]")
PAD_TOKEN_ID = tokenizer.token_to_id("[PAD]")
IGNORE_INDEX = -100 # PyTorch standard for ignoring loss

# ==========================================
# 2. Formatting & Tokenization Logic
# ==========================================
def process_gsm8k_nar(examples):
    """Processes GSM8K directly into padded/masked NAR tensors."""
    inputs, labels = [], []
    
    for question, answer in zip(examples["question"], examples["answer"]):
        # 1. Split and clean
        parts = answer.split("####")
        raw_reasoning = parts[0].strip()
        clean_reasoning = re.sub(r'<<.*?>>', '', raw_reasoning)
        final_answer = parts[1].strip() if len(parts) > 1 else ""
        
        # 2. Define the Prompt vs the Full Target
        prompt_text = f"[INST] {question} [/INST]\n"
        full_text = (
            f"{prompt_text}"
            f"[THINK]\n{clean_reasoning}\n[/THINK]\n"
            f"[ANSWER] {final_answer} [/ANSWER]"
        )
        
        # 3. Tokenize separately
        prompt_ids = tokenizer.encode(prompt_text).ids[:MAX_SEQ_LEN]
        full_ids = tokenizer.encode(full_text).ids[:MAX_SEQ_LEN]
        
        # 4. Pad prompt with [MASK] to create x_input
        x_input = prompt_ids + [MASK_TOKEN_ID] * (MAX_SEQ_LEN - len(prompt_ids))
        
        # 5. Pad full_text with IGNORE_INDEX to create y_true
        y_true = full_ids + [IGNORE_INDEX] * (MAX_SEQ_LEN - len(full_ids))
        
        inputs.append(x_input)
        labels.append(y_true)
        
    return {"input_ids": inputs, "labels": labels}

def process_tinystories_nar(examples):
    """Processes TinyStories. We extract the first few words as a prompt."""
    inputs, labels = [], []
    
    for text in examples["text"]:
        # 1. Define the full target
        full_ids = tokenizer.encode(text).ids[:MAX_SEQ_LEN]
        
        # 2. Create a prompt (e.g., the first 8 tokens to kickstart generation)
        # If you want unconditional generation, just use an empty list or [BOS] token here.
        prompt_ids = full_ids[:8] 
        
        # 3. Pad prompt with [MASK] to create x_input
        x_input = prompt_ids + [MASK_TOKEN_ID] * (MAX_SEQ_LEN - len(prompt_ids))
        
        # 4. Pad full_text with IGNORE_INDEX to create y_true
        y_true = full_ids + [IGNORE_INDEX] * (MAX_SEQ_LEN - len(full_ids))
        
        inputs.append(x_input)
        labels.append(y_true)

    return {"input_ids": inputs, "labels": labels}

# ==========================================
# 3. Execution Pipeline
# ==========================================
print("Loading Datasets...")
gsm8k_dataset = load_dataset("gsm8k", "main", split="train")
tinystories_dataset = load_dataset("roneneldan/TinyStories", split="train[:100000]")

print("Processing into NAR Tensors... (This merges formatting and tokenizing)")
# Map the new NAR processing functions directly
gsm8k_tokenized = gsm8k_dataset.map(
    process_gsm8k_nar, 
    batched=True, 
    batch_size=1000, 
    remove_columns=gsm8k_dataset.column_names # Strip all old columns
)

tinystories_tokenized = tinystories_dataset.map(
    process_tinystories_nar, 
    batched=True, 
    batch_size=1000, 
    remove_columns=tinystories_dataset.column_names # Strip all old columns
)

print("Formatting for PyTorch...")
gsm8k_tokenized.set_format(type="torch", columns=["input_ids", "labels"])
tinystories_tokenized.set_format(type="torch", columns=["input_ids", "labels"])

# Now, a single batch gives you perfectly sized [B, 512] tensors ready for TRM!

Loading Tokenizer...
Loading Datasets...
Processing into NAR Tensors... (This merges formatting and tokenizing)
Formatting for PyTorch...


In [4]:
tokenizer.decode(gsm8k_tokenized[0]['input_ids'].tolist())


'Natalia sold clips to 4 8 of her friends in April, and then she sold half as many clips in May. How many clips did Natalia sell altogether in April and May?'

In [5]:
fix = torch.where(gsm8k_tokenized[0]['labels'] != -100,gsm8k_tokenized[0]['labels'],0)
tokenizer.decode(fix.tolist())

'Natalia sold clips to 4 8 of her friends in April, and then she sold half as many clips in May. How many clips did Natalia sell altogether in April and May? Natalia sold 4 8 / 2 = 2 4 clips in May. Natalia sold 4 8 + 2 4 = 7 2 clips altogether in April and May. 7 2'

In [6]:
gsm8k_tokenized[0]['input_ids'].tolist()

[2,
 5,
 58,
 360,
 449,
 750,
 2592,
 539,
 3721,
 346,
 32,
 36,
 403,
 375,
 528,
 383,
 45,
 194,
 191,
 357,
 24,
 344,
 640,
 395,
 2592,
 3189,
 464,
 830,
 539,
 3721,
 383,
 57,
 354,
 26,
 1840,
 830,
 539,
 3721,
 509,
 58,
 360,
 449,
 750,
 2795,
 552,
 737,
 600,
 383,
 45,
 194,
 191,
 357,
 344,
 57,
 354,
 43,
 6,
 3,
 4,
 4,
 4,
 4,
 4,
 4,
 4,
 4,
 4,
 4,
 4,
 4,
 4,
 4,
 4,
 4,
 4,
 4,
 4,
 4,
 4,
 4,
 4,
 4,
 4,
 4,
 4,
 4,
 4,
 4,
 4,
 4,
 4,
 4,
 4,
 4,
 4,
 4,
 4,
 4,
 4,
 4,
 4,
 4,
 4,
 4,
 4,
 4,
 4,
 4,
 4,
 4,
 4,
 4,
 4,
 4,
 4,
 4,
 4,
 4,
 4,
 4,
 4,
 4,
 4,
 4,
 4,
 4,
 4,
 4,
 4,
 4,
 4,
 4,
 4,
 4,
 4,
 4,
 4,
 4,
 4,
 4,
 4,
 4,
 4,
 4,
 4,
 4,
 4,
 4,
 4,
 4,
 4,
 4,
 4,
 4,
 4,
 4,
 4,
 4,
 4,
 4,
 4,
 4,
 4,
 4,
 4,
 4,
 4,
 4,
 4,
 4,
 4,
 4,
 4,
 4,
 4,
 4,
 4,
 4,
 4,
 4,
 4,
 4,
 4,
 4,
 4,
 4,
 4,
 4,
 4,
 4,
 4,
 4,
 4,
 4,
 4,
 4,
 4,
 4,
 4,
 4,
 4,
 4,
 4,
 4,
 4,
 4,
 4,
 4,
 4,
 4,
 4,
 4,
 4,
 4,
 4,
 4,
 4,
 4,
 4,
 4,
 4,
 4,
 4,
 4,

In [7]:
from torch.utils.data import DataLoader

gsm8k_loader = DataLoader(gsm8k_tokenized, batch_size=BATCH_SIZE, shuffle=True)
tinystories_loader = DataLoader(tinystories_tokenized, batch_size=BATCH_SIZE, shuffle=True)

print("\n--- Pipeline Complete! ---")
print(f"GSM8K Batches available: {len(gsm8k_loader)}")
print(f"TinyStories Batches available: {len(tinystories_loader)}")

# Let's verify a batch!
sample_batch = next(iter(gsm8k_loader))
print(f"\nShape of input_ids: {sample_batch['input_ids'].shape}")
print(f"Shape of labels: {sample_batch['labels'].shape}")
print(f"First 10 input_ids: {sample_batch['input_ids'][0][:10]}")
print(f"Last 10 labels (Notice the -100 padding!): {sample_batch['labels'][0][-10:]}")


--- Pipeline Complete! ---
GSM8K Batches available: 468
TinyStories Batches available: 6250

Shape of input_ids: torch.Size([16, 512])
Shape of labels: torch.Size([16, 512])
First 10 input_ids: tensor([   2,    5, 3424,  770,  459, 2126,   36, 1186, 3729, 3228])
Last 10 labels (Notice the -100 padding!): tensor([-100, -100, -100, -100, -100, -100, -100, -100, -100, -100])


In [18]:
config_dict = {
    # 1. Sequence & Vocabulary (The Generative Shift)
    # Must match training micro-batch if puzzle_emb_ndim > 0 (CastedSparseEmbedding); keep aligned with BATCH_SIZE
    "batch_size": 16,
    "seq_len": 512,                 # Maximum context window for text
    "vocab_size": 4096,             # Must match your Micro-BPE tokenizer!
    
    # 2. Transformer Core (~7M Params)
    "hidden_size": 512,             
    "expansion": 4.0,               
    "num_heads": 8,                 
    "pos_encodings": "rope",        # CRITICAL: Replaces 'none' from Sudoku
    
    # 3. Recursion & Loops
    "L_layers": 2,                  # Physical layers
    "H_layers": 1,                  # (Ignored)
    "L_cycles": 2,                  # n
    "H_cycles": 8,                 # T
    
    # 4. Adaptive Computation Time (ACT)
    "halt_max_steps": 16,           # Nsupp
    "halt_exploration_prob": 0.1,   
    
    # 5. Stability & Normalization
    "rms_norm_eps": 1e-5,
    "rope_theta": 10000.0,
    "forward_dtype": "bfloat16",    
    
    # 6. Puzzle Disablement (Turning off Sudoku features)
    "mlp_t": False,                 # CRITICAL: Must be False to use Attention
    "puzzle_emb_ndim": 0,           # Turn off puzzle embeddings
    "num_puzzle_identifiers": 1,    # Safe dummy value
    "puzzle_emb_len": 0,
    "no_ACT_continue": True, 
    "causal": False    
}

In [19]:
import torch
import torch.nn.functional as F
from torch.optim import AdamW
import math


def cosine_schedule_with_warmup_lr_lambda(
    current_step: int,
    *,
    base_lr: float,
    num_warmup_steps: int,
    num_training_steps: int,
    min_ratio: float = 0.0,
    num_cycles: float = 0.5,
):
    """Same schedule as pretrain.py: linear warmup then cosine decay."""
    current_step = min(max(int(current_step), 0), int(num_training_steps))
    if current_step < num_warmup_steps:
        return base_lr * float(current_step) / float(max(1, num_warmup_steps))
    progress = float(current_step - num_warmup_steps) / float(max(1, num_training_steps - num_warmup_steps))
    return base_lr * (min_ratio + max(0.0, (1 - min_ratio) * 0.5 * (1.0 + math.cos(math.pi * float(num_cycles) * 2.0 * progress))))


from upload import TinyRecursiveReasoningModel_ACTV1
from ema import EMAHelper

# Assume your model and loaders are imported/defined above
# from trm_singlez import TinyRecursiveReasoningModel_ACTV1, TinyRecursiveReasoningModel_ACTV1Config
# gsm8k_loader, tinystories_loader = ... 

# ==========================================
# 1. Initialization & VRAM Optimization
# ==========================================
import gc

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Training on device: {device}")

# 4 GB GPUs: free cached allocations from earlier cells before moving weights to CUDA
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()
    free_b, total_b = torch.cuda.mem_get_info()
    print(f"CUDA mem_get_info free/total MiB: {free_b // 2**20:.0f} / {total_b // 2**20:.0f}")
    if free_b < 300 * 1024**2:
        print("Very little free VRAM — close other GPU apps or Kernel > Restart, then run from the top.")

# Initialize Model (Make sure config has causal=True and vocab_size=4096)
model = TinyRecursiveReasoningModel_ACTV1(config_dict).to(device)

# Optimizer LR is set each update step (see train_epoch + lr_schedule), matching pretrain.py style.
optimizer = AdamW(model.parameters(), lr=0.0, weight_decay=0.1)

# Mixed Precision Scaler (Crucial for 4GB/8GB GPUs)
scaler = torch.amp.GradScaler('cuda')

# Gradient Accumulation: Simulates a larger batch size without crashing VRAM
accumulation_steps = 4

# LR schedule hyperparams (pretrain.py: lr, lr_warmup_steps, lr_min_ratio)
base_lr_phase1 = 1e-4
base_lr_phase2 = 1e-4
lr_warmup_steps = 2000
lr_min_ratio = 0.0
lr_schedule = {
    "step": 0,
    "total_steps": 1,
    "base_lr": base_lr_phase1,
    "warmup_steps": 1,
    "min_ratio": lr_min_ratio,
}

# EMA (same flags / helper as pretrain.py PretrainConfig.ema, ema_rate)
ema = False
ema_rate = 0.999
ema_helper = None
if ema:
    print("Setup EMA")
    ema_helper = EMAHelper(mu=ema_rate)
    ema_helper.register(model)


def save_training_checkpoint(path, model, ema_helper):
    """Save EMA weights when EMA is enabled (mirrors pretrain checkpointing with ema_copy)."""
    if ema_helper is not None:
        ema_model = ema_helper.ema_copy(model)
        try:
            torch.save(ema_model.state_dict(), path)
        finally:
            del ema_model
    else:
        torch.save(model.state_dict(), path)


Training on device: cuda
CUDA mem_get_info free/total MiB: 2647 / 4095


In [20]:
!nvidia-smi

Sat Apr 18 19:47:12 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 566.14                 Driver Version: 566.14         CUDA Version: 12.7     |
|-----------------------------------------+------------------------+----------------------+
| GPU  Name                  Driver-Model | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA GeForce RTX 2050      WDDM  |   00000000:01:00.0 Off |                  N/A |
| N/A   72C    P5              5W /   45W |     976MiB /   4096MiB |      7%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [23]:
import torch
import torch.nn.functional as F

def train_epoch(model, dataloader, optimizer, scaler, epoch, phase_name, ema_helper=None, lr_schedule=None):
    model.train()
    total_loss = 0
    
    # Ensure these are globally accessible or pass them in
    mask_token_id = tokenizer.token_to_id("[MASK]")
    pad_token_id = tokenizer.token_to_id("[PAD]")
    
    for batch_idx, batch in enumerate(dataloader):
        input_ids = batch["input_ids"].to(device)
        
        # ---------------------------------------------------------
        # BREAKTHROUGH 1: The Padding Leak Fix
        # ---------------------------------------------------------
        # Clone labels so we can modify them without breaking the dataset.
        # Change the label from -100 to pad_token_id so the model explicitly 
        # learns boundary control and sequence termination.
        labels = batch["labels"].clone().to(device)
        is_pad = (input_ids == pad_token_id)
        labels[is_pad] = pad_token_id

        # ---------------------------------------------------------
        # BREAKTHROUGH 4: Curriculum Depth (Recursive Warmup)
        # ---------------------------------------------------------
        # Prevent "Infinite Mirror" activation blowouts on fresh weights
        # by gradually increasing the number of detached loops.
        if epoch == 1:
            if batch_idx < 500:
                model.config.H_cycles = 1  # 2 effective layers (Learn stability)
            elif batch_idx < 1000:
                model.config.H_cycles = 2  # 4 effective layers
            elif batch_idx < 1500:
                model.config.H_cycles = 4  # 8 effective layers
            else:
                model.config.H_cycles = 8  # Full 16-layer capacity
        else:
            model.config.H_cycles = 8 # Full capacity for remaining epochs

        # Ensure L_cycles remains safely at 2 for the backward pass
        model.config.L_cycles = 2
        
        # ---------------------------------------------------------
        # BREAKTHROUGH 2: Curriculum Masking
        # ---------------------------------------------------------
        # Gradually increase task difficulty to prevent gradient whiplash 
        # and allow the 42-layer effective depth to learn N-gram syntax first.
        if epoch <= 2:
            max_mask = 0.40  # Phase 1: Local N-gram grammar
        elif epoch == 3:
            max_mask = 0.70  # Phase 2: Sentence-level structure
        else:
            max_mask = 1.00  # Phase 3: Global sequence generation
            
        mask_ratio = torch.empty(1).uniform_(0.1, max_mask).item()
        
        prob_matrix = torch.full(labels.shape, mask_ratio, device=device)
        
        # Protect the GSM8K/TinyStories prompt (which is STILL -100) from masking.
        prob_matrix.masked_fill_(labels == -100, 0.0) 
        masked_indices = torch.bernoulli(prob_matrix).bool()
        
        masked_inputs = input_ids.clone()
        masked_inputs[masked_indices] = mask_token_id
        
        # ---------------------------------------------------------
        # BREAKTHROUGH 3: The Identity Collapse Fix
        # ---------------------------------------------------------
        # Ignore the loss for unmasked tokens. If we don't do this, the model 
        # will learn to turn off its bidirectional attention to perfectly copy 
        # the unmasked tokens, completely destroying its ability to reason.
        labels[~masked_indices] = -100
        
        # Prepare the TRM specific batch dictionary
        trm_batch = {
            "inputs": masked_inputs,
            "puzzle_identifiers": torch.zeros((input_ids.shape[0], model.config.num_puzzle_identifiers), dtype=torch.int32, device=device)
        }
        carry = model.initial_carry(trm_batch)
        
        # Forward Pass with Mixed Precision
        with torch.amp.autocast('cuda', dtype=torch.bfloat16):
            _, outputs = model(carry, trm_batch)
            logits = outputs["logits"]
            
            loss = F.cross_entropy(
                logits.view(-1, model.config.vocab_size), 
                labels.view(-1)
            )
            loss = loss / accumulation_steps

        # Backward Pass
        scaler.scale(loss).backward()
        
        # Optimizer Step & LR Scheduling
        if (batch_idx + 1) % accumulation_steps == 0:
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            
            # Update learning rate if a schedule is provided
            if lr_schedule is not None:
                lr_schedule["step"] += 1
                s = lr_schedule["step"]
                lr_this = cosine_schedule_with_warmup_lr_lambda(
                    s,
                    base_lr=lr_schedule["base_lr"],
                    num_warmup_steps=lr_schedule["warmup_steps"],
                    num_training_steps=lr_schedule["total_steps"],
                    min_ratio=lr_schedule["min_ratio"],
                )
                for param_group in optimizer.param_groups:
                    param_group["lr"] = lr_this
                    
            scaler.step(optimizer)
            scaler.update()
            optimizer.zero_grad()

        # Logging & Metrics
        total_loss += loss.item() * accumulation_steps
        if batch_idx % 50 == 0:
            print(f"Epoch {epoch} [{phase_name}] | Batch {batch_idx}/{len(dataloader)} | Mask: {mask_ratio*100:.0f}% | Loss: {loss.item() * accumulation_steps:.4f}")

        # Exponential Moving Average Update
        if ema_helper is not None:
            ema_helper.update(model)

    return total_loss / len(dataloader)

only tinystoreis

In [24]:
num_tinystories_epochs = 2

def _optimizer_steps_per_phase(loader, accum, epochs):
    """Optimizer steps per phase (matches (batch_idx+1) % accum == 0 counting)."""
    return max(1, (len(loader) // accum) * epochs)

print("\n=== PHASE 1: Syntax (TinyStories) ===")
# Fresh warmup + cosine decay for Phase 1
lr_schedule["step"] = 0
lr_schedule["base_lr"] = base_lr_phase1  # Make sure this is now set to 1e-4 in your config!
lr_schedule["total_steps"] = _optimizer_steps_per_phase(tinystories_loader, accumulation_steps, num_tinystories_epochs)
lr_schedule["warmup_steps"] = min(lr_warmup_steps, lr_schedule["total_steps"])
lr_schedule["min_ratio"] = lr_min_ratio

for epoch in range(1, num_tinystories_epochs + 1):
    # This will call our newly updated train_epoch with Curriculum Masking & BPTT fixes
    avg_loss = train_epoch(model, tinystories_loader, optimizer, scaler, epoch, "TinyStories", ema_helper, lr_schedule)
    print(f"--> Phase 1, Epoch {epoch} Complete. Avg Loss: {avg_loss:.4f}\n")

    # Save Phase 1 weights 
    save_training_checkpoint(f"g_trm_tinystories_epoch_{epoch}.pt", model, ema_helper)

print("TinyStories Foundation Training Complete!")


=== PHASE 1: Syntax (TinyStories) ===
Epoch 1 [TinyStories] | Batch 0/6250 | Mask: 36% | Loss: 6.2308
Epoch 1 [TinyStories] | Batch 50/6250 | Mask: 22% | Loss: 5.8052
Epoch 1 [TinyStories] | Batch 100/6250 | Mask: 25% | Loss: 5.9769
Epoch 1 [TinyStories] | Batch 150/6250 | Mask: 18% | Loss: 5.7101
Epoch 1 [TinyStories] | Batch 200/6250 | Mask: 18% | Loss: 5.9586
Epoch 1 [TinyStories] | Batch 250/6250 | Mask: 39% | Loss: 5.8116
Epoch 1 [TinyStories] | Batch 300/6250 | Mask: 14% | Loss: 5.6292
Epoch 1 [TinyStories] | Batch 350/6250 | Mask: 19% | Loss: 5.8361
Epoch 1 [TinyStories] | Batch 400/6250 | Mask: 35% | Loss: 5.9083
Epoch 1 [TinyStories] | Batch 450/6250 | Mask: 35% | Loss: 5.7338
Epoch 1 [TinyStories] | Batch 500/6250 | Mask: 11% | Loss: 5.9157
Epoch 1 [TinyStories] | Batch 550/6250 | Mask: 14% | Loss: 5.7440
Epoch 1 [TinyStories] | Batch 600/6250 | Mask: 29% | Loss: 5.6737
Epoch 1 [TinyStories] | Batch 650/6250 | Mask: 37% | Loss: 5.7887
Epoch 1 [TinyStories] | Batch 700/6250 |

KeyboardInterrupt: 

In [12]:

num_tinystories_epochs = 2
num_gsm8k_epochs = 5


def _optimizer_steps_per_phase(loader, accum, epochs):
    """Optimizer steps per phase (matches (batch_idx+1) % accum == 0 counting)."""
    return max(1, (len(loader) // accum) * epochs)


print("\n=== PHASE 1: Syntax (TinyStories) ===")
lr_schedule["step"] = 0
lr_schedule["base_lr"] = base_lr_phase1
lr_schedule["total_steps"] = _optimizer_steps_per_phase(tinystories_loader, accumulation_steps, num_tinystories_epochs)
lr_schedule["warmup_steps"] = min(lr_warmup_steps, lr_schedule["total_steps"])
lr_schedule["min_ratio"] = lr_min_ratio

for epoch in range(1, num_tinystories_epochs + 1):
    avg_loss = train_epoch(model, tinystories_loader, optimizer, scaler, epoch, "TinyStories", ema_helper, lr_schedule)
    print(f"--> Phase 1, Epoch {epoch} Complete. Avg Loss: {avg_loss:.4f}\n")

    # NEW: Save Phase 1 weights (EMA copy when ema=True, same as pretrain checkpointing)
    save_training_checkpoint(f"g_trm_tinystories_epoch_{epoch}.pt", model, ema_helper)

print("\n=== PHASE 2: Reasoning (GSM8K) ===")
# New phase: fresh warmup + cosine to base_lr_phase2 (same pattern as pretrain schedule)
lr_schedule["step"] = 0
lr_schedule["base_lr"] = base_lr_phase2
lr_schedule["total_steps"] = _optimizer_steps_per_phase(gsm8k_loader, accumulation_steps, num_gsm8k_epochs)
lr_schedule["warmup_steps"] = min(lr_warmup_steps, lr_schedule["total_steps"])
lr_schedule["min_ratio"] = lr_min_ratio

for epoch in range(1, num_gsm8k_epochs + 1):
    avg_loss = train_epoch(model, gsm8k_loader, optimizer, scaler, epoch, "GSM8K", ema_helper, lr_schedule)
    print(f"--> Phase 2, Epoch {epoch} Complete. Avg Loss: {avg_loss:.4f}\n")
    
    # Save checkpoint
    save_training_checkpoint(f"g_trm_gsm8k_epoch_{epoch}.pt", model, ema_helper)

print("Training Complete!")


=== PHASE 1: Syntax (TinyStories) ===
Epoch 1 [TinyStories] | Batch 0/6250 | Mask: 22% | Loss: 8.8093
Epoch 1 [TinyStories] | Batch 50/6250 | Mask: 12% | Loss: 8.4826
Epoch 1 [TinyStories] | Batch 100/6250 | Mask: 40% | Loss: 7.8753
Epoch 1 [TinyStories] | Batch 150/6250 | Mask: 13% | Loss: 7.4429
Epoch 1 [TinyStories] | Batch 200/6250 | Mask: 39% | Loss: 7.0655
Epoch 1 [TinyStories] | Batch 250/6250 | Mask: 20% | Loss: 6.9351
Epoch 1 [TinyStories] | Batch 300/6250 | Mask: 12% | Loss: 6.6718
Epoch 1 [TinyStories] | Batch 350/6250 | Mask: 29% | Loss: 6.6778
Epoch 1 [TinyStories] | Batch 400/6250 | Mask: 19% | Loss: 6.5706
Epoch 1 [TinyStories] | Batch 450/6250 | Mask: 25% | Loss: 6.4181
Epoch 1 [TinyStories] | Batch 500/6250 | Mask: 28% | Loss: 6.2342
Epoch 1 [TinyStories] | Batch 550/6250 | Mask: 18% | Loss: 6.3140
Epoch 1 [TinyStories] | Batch 600/6250 | Mask: 22% | Loss: 6.1602
Epoch 1 [TinyStories] | Batch 650/6250 | Mask: 21% | Loss: 6.1731
Epoch 1 [TinyStories] | Batch 700/6250 |

KeyboardInterrupt: 

In [ ]:
import torch
import re
from datasets import load_dataset
from tokenizers import Tokenizer
import importlib
import upload as _upload_mod

# Jupyter caches imports; reload so `upload.py` changes (e.g. `generate`) apply without a kernel restart.
importlib.reload(_upload_mod)
from upload import TinyRecursiveReasoningModel_ACTV1

# ==========================================
# 1. Initialization
# ==========================================
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Initializing Evaluation Pipeline...")

# Load your custom tokenizer
tokenizer = Tokenizer.from_file("./trm_tokenizer/tokenizer_mask.json")

# Load model (Use the exact config_dict from your training script)
model = TinyRecursiveReasoningModel_ACTV1(config_dict)
model.load_state_dict(torch.load("g_trm_gsm8k_epoch_5.pt", map_location=device))
model.to(device)
model.eval()

# Load the GSM8K Test Split (Not the train split!)
test_dataset = load_dataset("gsm8k", "main", split="test")

# ==========================================
# 2. Answer Extraction Logic
# ==========================================
def extract_model_answer(generated_text: str):
    """Uses Regex to find the number inside the [ANSWER] tags."""
    match = re.search(r'\[ANSWER\](.*?)\[/ANSWER\]', generated_text, re.DOTALL)
    if match:
        numbers = re.findall(r'-?\d+\.?\d*', match.group(1))
        return float(numbers[-1]) if numbers else None
    return None

def extract_ground_truth(raw_answer: str):
    """Extracts the final number after #### in the GSM8K dataset."""
    parts = raw_answer.split("####")
    if len(parts) > 1:
        numbers = re.findall(r'-?\d+\.?\d*', parts[1])
        return float(numbers[-1]) if numbers else None
    return None

# ==========================================
# 3. The Evaluation Loop
# ==========================================
total_questions = 0
correct_answers = 0

# Dictionaries to track the "Inference Gap"
difficulty_tracker = {"Easy (1-2 steps)": [0, 0], "Medium (3-4 steps)": [0, 0], "Hard (5+ steps)": [0, 0]}

print(f"Starting evaluation on {len(test_dataset)} test questions...\n")

for i, example in enumerate(test_dataset):
    question = example["question"]
    ground_truth_text = example["answer"]
    
    num_steps = ground_truth_text.split("####")[0].count('\n')
    if num_steps <= 2:
        difficulty_category = "Easy (1-2 steps)"
    elif num_steps <= 4:
        difficulty_category = "Medium (3-4 steps)"
    else:
        difficulty_category = "Hard (5+ steps)"

    prompt = f"[INST] {question} [/INST]\n[THINK]"
    input_ids = torch.tensor([tokenizer.encode(prompt).ids], dtype=torch.long).to(device)

    # Generate with strict temperature and NO repetition penalty
    with torch.no_grad():
        output_ids = model.generate(input_ids, max_new_tokens=100, tokenizer=tokenizer)
    
    generated_text = tokenizer.decode(output_ids[0].tolist(), skip_special_tokens=False)
    
    model_ans = extract_model_answer(generated_text)
    truth_ans = extract_ground_truth(ground_truth_text)

    # ==========================================
    # INDENTATION FIXED BELOW
    # ==========================================
    total_questions += 1
    difficulty_tracker[difficulty_category][1] += 1
    
    is_correct = False
    if model_ans is not None and truth_ans is not None:
        if model_ans == truth_ans:
            correct_answers += 1
            difficulty_tracker[difficulty_category][0] += 1
            is_correct = True

    # if i % 10 == 0 or model_ans is None: 
    print(f"\n[{i}/{len(test_dataset)}] Accuracy so far: {(correct_answers/total_questions)*100:.2f}%")
    if not is_correct:
        print(f"  -> Expected: {truth_ans}")
        print(f"  -> Extracted: {model_ans}")
        print(f"  -> RAW MODEL OUTPUT:\n{generated_text}\n")
        print("-" * 40)

# ==========================================
# 4. Final Thesis Metrics Output
# ==========================================
final_accuracy = (correct_answers / total_questions) * 100
print("\n" + "="*50)
print(f"FINAL MODEL ACCURACY: {final_accuracy:.2f}%")
print("="*50)
print("INFERENCE GAP ANALYSIS (For Presentation Graph):")
for category, counts in difficulty_tracker.items():
    correct, total = counts
    if total > 0:
        acc = (correct / total) * 100
        print(f"- {category}: {acc:.2f}% ({correct}/{total})")

In [34]:
import torch
import importlib
import upload as _upload_mod

# Jupyter caches imports; reload so `upload.py` changes (e.g. `generate`) apply without a kernel restart.
importlib.reload(_upload_mod)
from upload import TinyRecursiveReasoningModel_ACTV1

from tokenizers import Tokenizer

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Loading Epoch 1 Checkpoint...")

# 1. Load the tokenizer and Model
tokenizer = Tokenizer.from_file("/workspace/G-TRM-NAR-/trm_tokenizer/tokenizer_mask_1.json")
model = TinyRecursiveReasoningModel_ACTV1(config_dict)
model.load_state_dict(torch.load("/workspace/G-TRM-NAR-/g_trm_tinystories_epoch_1.pt", map_location=device))
model.to(device)
model.eval()

# 2. A simple TinyStories Prompt
prompt = "[BOS] Once upon a time, there was a little girl named Lily. She loved to play in the"
input_ids = torch.tensor([tokenizer.encode(prompt).ids], dtype=torch.long).to(device)

# 3. Generate (Keep it to 100 tokens to see immediate latent behavior)
print("Generating parallel sequence...")
with torch.no_grad():
    output_ids = model.generate(input_ids, max_new_tokens=100, iterations=30, tokenizer=tokenizer)
    
generated_text = tokenizer.decode(output_ids[0].tolist(), skip_special_tokens=False)

print("\n--- MODEL OUTPUT ---")
print(generated_text)

Loading Epoch 1 Checkpoint...
Generating parallel sequence...



--- MODEL OUTPUT ---
. friends friends friends Once two two friends friends friends friends two friends best friends. friends friends two. best friends. friends. Once time. friends. two friends friends friends friends two two friends best. friends best two friends friends Once friends two two friends friends two friends friends two friends friends friends friends friends friends two friends friends friends friends friends friends friends friends Once best friends friends friends friends friends friends friends friends Once friends friends time friends two two friends friends friends friends friends friends friends best friends friends friends friends friends


Curiculum Masking

increase the batch size

In [10]:
import torch
from datasets import load_dataset
from tokenizers import Tokenizer
import re

# ==========================================
# 1. Configuration & Tokenizer Setup
# ==========================================
MAX_SEQ_LEN = 512 
# DataLoader micro-batch; use 8–16 on 4 GB GPUs (64 × 512 activations OOMs easily during training)
BATCH_SIZE = 8

print("Loading Tokenizer...")
tokenizer = Tokenizer.from_file("C:/Users/MSI-1/Desktop/adnan_final/trm_tokenizer/tokenizer_mask_1.json")

# Extract special token IDs (assuming your tokenizer has these defined)
MASK_TOKEN_ID = tokenizer.token_to_id("[MASK]")
PAD_TOKEN_ID = tokenizer.token_to_id("[PAD]")
IGNORE_INDEX = -100 # PyTorch standard for ignoring loss

# ==========================================
# 2. Formatting & Tokenization Logic
# ==========================================
def process_gsm8k_nar(examples):
    """Processes GSM8K directly into padded/masked NAR tensors."""
    inputs, labels = [], []
    
    for question, answer in zip(examples["question"], examples["answer"]):
        # 1. Split and clean
        parts = answer.split("####")
        raw_reasoning = parts[0].strip()
        clean_reasoning = re.sub(r'<<.*?>>', '', raw_reasoning)
        final_answer = parts[1].strip() if len(parts) > 1 else ""
        
        # 2. Define the Prompt vs the Full Target
        prompt_text = f"[INST] {question} [/INST]\n"
        full_text = (
            f"{prompt_text}"
            f"[THINK]\n{clean_reasoning}\n[/THINK]\n"
            f"[ANSWER] {final_answer} [/ANSWER]"
        )
        
        # 3. Tokenize separately
        prompt_ids = tokenizer.encode(prompt_text).ids[:MAX_SEQ_LEN]
        full_ids = tokenizer.encode(full_text).ids[:MAX_SEQ_LEN]
        
        # 4. Pad prompt with [MASK] to create x_input
        x_input = prompt_ids + [MASK_TOKEN_ID] * (MAX_SEQ_LEN - len(prompt_ids))
        
        # 5. Pad full_text with IGNORE_INDEX to create y_true
        y_true = full_ids + [IGNORE_INDEX] * (MAX_SEQ_LEN - len(full_ids))
        
        inputs.append(x_input)
        labels.append(y_true)
        
    return {"input_ids": inputs, "labels": labels}

def process_tinystories_nar(examples):
    """Processes TinyStories. We extract the first few words as a prompt."""
    inputs, labels = [], []
    
    for text in examples["text"]:
        # 1. Define the full target
        full_ids = tokenizer.encode(text).ids[:MAX_SEQ_LEN]
        
        # 2. Create a prompt (e.g., the first 8 tokens to kickstart generation)
        # If you want unconditional generation, just use an empty list or [BOS] token here.
        prompt_ids = full_ids[:8] 
        
        # 3. Pad prompt with [MASK] to create x_input
        x_input = prompt_ids + [MASK_TOKEN_ID] * (MAX_SEQ_LEN - len(prompt_ids))
        
        # 4. Pad full_text with IGNORE_INDEX to create y_true
        y_true = full_ids + [IGNORE_INDEX] * (MAX_SEQ_LEN - len(full_ids))
        
        inputs.append(x_input)
        labels.append(y_true)

    return {"input_ids": inputs, "labels": labels}

# ==========================================
# 3. Execution Pipeline
# ==========================================
print("Loading Datasets...")
gsm8k_dataset = load_dataset("gsm8k", "main", split="train")
tinystories_dataset = load_dataset("roneneldan/TinyStories", split="train[:100000]")

print("Processing into NAR Tensors... (This merges formatting and tokenizing)")
# Map the new NAR processing functions directly
gsm8k_tokenized = gsm8k_dataset.map(
    process_gsm8k_nar, 
    batched=True, 
    batch_size=1000, 
    remove_columns=gsm8k_dataset.column_names # Strip all old columns
)

tinystories_tokenized = tinystories_dataset.map(
    process_tinystories_nar, 
    batched=True, 
    batch_size=1000, 
    remove_columns=tinystories_dataset.column_names # Strip all old columns
)

print("Formatting for PyTorch...")
gsm8k_tokenized.set_format(type="torch", columns=["input_ids", "labels"])
tinystories_tokenized.set_format(type="torch", columns=["input_ids", "labels"])

# Now, a single batch gives you perfectly sized [B, 512] tensors ready for TRM!

Loading Tokenizer...
Loading Datasets...
Processing into NAR Tensors... (This merges formatting and tokenizing)


Map: 100%|██████████| 100000/100000 [00:33<00:00, 3010.68 examples/s]

Formatting for PyTorch...


In [11]:
from torch.utils.data import DataLoader

gsm8k_loader = DataLoader(gsm8k_tokenized, batch_size=BATCH_SIZE, shuffle=True)
tinystories_loader = DataLoader(tinystories_tokenized, batch_size=BATCH_SIZE, shuffle=True)

print("\n--- Pipeline Complete! ---")
print(f"GSM8K Batches available: {len(gsm8k_loader)}")
print(f"TinyStories Batches available: {len(tinystories_loader)}")

# Let's verify a batch!
sample_batch = next(iter(gsm8k_loader))
print(f"\nShape of input_ids: {sample_batch['input_ids'].shape}")
print(f"Shape of labels: {sample_batch['labels'].shape}")
print(f"First 10 input_ids: {sample_batch['input_ids'][0][:10]}")
print(f"Last 10 labels (Notice the -100 padding!): {sample_batch['labels'][0][-10:]}")


--- Pipeline Complete! ---
GSM8K Batches available: 117
TinyStories Batches available: 1563

Shape of input_ids: torch.Size([64, 512])
Shape of labels: torch.Size([64, 512])
First 10 input_ids: tensor([   2,    5,   60,  199, 1673,  192, 1034,   29,   32,  613])
Last 10 labels (Notice the -100 padding!): tensor([-100, -100, -100, -100, -100, -100, -100, -100, -100, -100])


In [12]:
config_dict = {
    # 1. Sequence & Vocabulary (The Generative Shift)
    # Must match training micro-batch if puzzle_emb_ndim > 0 (CastedSparseEmbedding); keep aligned with BATCH_SIZE
    "batch_size": 8,
    "seq_len": 512,                 # Maximum context window for text
    "vocab_size": 4096,             # Must match your Micro-BPE tokenizer!
    
    # 2. Transformer Core (~7M Params)
    "hidden_size": 512,             
    "expansion": 4.0,               
    "num_heads": 8,                 
    "pos_encodings": "rope",        # CRITICAL: Replaces 'none' from Sudoku
    
    # 3. Recursion & Loops
    "L_layers": 2,                  # Physical layers
    "H_layers": 1,                  # (Ignored)
    "L_cycles": 6,                  # n
    "H_cycles": 1,                 # T
    
    # 4. Adaptive Computation Time (ACT)
    "halt_max_steps": 16,           # Nsupp
    "halt_exploration_prob": 0.1,   
    
    # 5. Stability & Normalization
    "rms_norm_eps": 1e-5,
    "rope_theta": 10000.0,
    "forward_dtype": "bfloat16",    
    
    # 6. Puzzle Disablement (Turning off Sudoku features)
    "mlp_t": False,                 # CRITICAL: Must be False to use Attention
    "puzzle_emb_ndim": 0,           # Turn off puzzle embeddings
    "num_puzzle_identifiers": 1,    # Safe dummy value
    "puzzle_emb_len": 0,
    "no_ACT_continue": True, 
    "causal": False    
}

In [15]:
import torch
import torch.nn.functional as F

def train_epoch(model, dataloader, optimizer, scaler, epoch, phase_name, ema_helper=None, lr_schedule=None):
    model.train()
    total_loss = 0
    
    # Ensure these are globally accessible or pass them in
    mask_token_id = tokenizer.token_to_id("[MASK]")
    pad_token_id = tokenizer.token_to_id("[PAD]")
    
    for batch_idx, batch in enumerate(dataloader):
        input_ids = batch["input_ids"].to(device)
        
        # ---------------------------------------------------------
        # BREAKTHROUGH 1: The Padding Leak Fix
        # ---------------------------------------------------------
        # Clone labels so we can modify them without breaking the dataset.
        # Change the label from -100 to pad_token_id so the model explicitly 
        # learns boundary control and sequence termination.
        labels = batch["labels"].clone().to(device)
        is_pad = (input_ids == pad_token_id)
        labels[is_pad] = pad_token_id
        
        # ---------------------------------------------------------
        # BREAKTHROUGH 2: Curriculum Masking
        # ---------------------------------------------------------
        # Gradually increase task difficulty to prevent gradient whiplash 
        # and allow the 42-layer effective depth to learn N-gram syntax first.
        if epoch <= 2:
            max_mask = 0.40  # Phase 1: Local N-gram grammar
        elif epoch == 3:
            max_mask = 0.70  # Phase 2: Sentence-level structure
        else:
            max_mask = 1.00  # Phase 3: Global sequence generation
            
        mask_ratio = torch.empty(1).uniform_(0.1, max_mask).item()
        
        prob_matrix = torch.full(labels.shape, mask_ratio, device=device)
        
        # Protect the GSM8K/TinyStories prompt (which is STILL -100) from masking.
        prob_matrix.masked_fill_(labels == -100, 0.0) 
        masked_indices = torch.bernoulli(prob_matrix).bool()
        
        masked_inputs = input_ids.clone()
        masked_inputs[masked_indices] = mask_token_id
        
        # ---------------------------------------------------------
        # BREAKTHROUGH 3: The Identity Collapse Fix
        # ---------------------------------------------------------
        # Ignore the loss for unmasked tokens. If we don't do this, the model 
        # will learn to turn off its bidirectional attention to perfectly copy 
        # the unmasked tokens, completely destroying its ability to reason.
        labels[~masked_indices] = -100
        
        # Prepare the TRM specific batch dictionary
        trm_batch = {
            "inputs": masked_inputs,
            "puzzle_identifiers": torch.zeros((input_ids.shape[0], model.config.num_puzzle_identifiers), dtype=torch.int32, device=device)
        }
        carry = model.initial_carry(trm_batch)
        
        # Forward Pass with Mixed Precision
        with torch.amp.autocast('cuda', dtype=torch.bfloat16):
            _, outputs = model(carry, trm_batch)
            logits = outputs["logits"]
            
            loss = F.cross_entropy(
                logits.view(-1, model.config.vocab_size), 
                labels.view(-1)
            )
            loss = loss / accumulation_steps

        # Backward Pass
        scaler.scale(loss).backward()
        
        # Optimizer Step & LR Scheduling
        if (batch_idx + 1) % accumulation_steps == 0:
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            
            # Update learning rate if a schedule is provided
            if lr_schedule is not None:
                lr_schedule["step"] += 1
                s = lr_schedule["step"]
                lr_this = cosine_schedule_with_warmup_lr_lambda(
                    s,
                    base_lr=lr_schedule["base_lr"],
                    num_warmup_steps=lr_schedule["warmup_steps"],
                    num_training_steps=lr_schedule["total_steps"],
                    min_ratio=lr_schedule["min_ratio"],
                )
                for param_group in optimizer.param_groups:
                    param_group["lr"] = lr_this
                    
            scaler.step(optimizer)
            scaler.update()
            optimizer.zero_grad()

        # Logging & Metrics
        total_loss += loss.item() * accumulation_steps
        if batch_idx % 50 == 0:
            print(f"Epoch {epoch} [{phase_name}] | Batch {batch_idx}/{len(dataloader)} | Mask: {mask_ratio*100:.0f}% | Loss: {loss.item() * accumulation_steps:.4f}")

        # Exponential Moving Average Update
        if ema_helper is not None:
            ema_helper.update(model)

    return total_loss / len(dataloader)

In [54]:
import torch
import torch.optim as optim
# Assuming tokenizer, dataloader, config_dict, and TinyRecursiveReasoningModel_ACTV1 are already in memory

print("Initializing Environment for Epoch 2...")
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# ==========================================
# 1. Load the Epoch 1 Checkpoint
# ==========================================
print("Loading model weights from Epoch 1...")
model = TinyRecursiveReasoningModel_ACTV1(config_dict)
model.load_state_dict(torch.load("g_trm_tinystories_epoch_1.pt", map_location=device))
model.to(device)

# ==========================================
# 2. Setup Optimizer, Scaler, and EMA
# ==========================================
# Re-initialize the optimizer
optimizer = optim.AdamW(model.parameters(), lr=5e-4, weight_decay=0.01)
scaler = torch.amp.GradScaler('cuda')

# (Optional) Re-initialize EMA if you are using a custom EMA helper class
# ema_helper = EMAHelper(mu=0.999) 
# ema_helper.register(model)
ema_helper = None # Replace with your EMA instance if active

# ==========================================
# 3. Resume Learning Rate Scheduler
# ==========================================
# We want to start at the step where Epoch 1 left off to avoid resetting the warmup
batches_per_epoch = len(tinystories_loader) # e.g., 15,625
accumulation_steps = 1 # Update if you are using grad accumulation

lr_schedule = {
    "step": batches_per_epoch // accumulation_steps, # FAST-FORWARD to the start of Epoch 2
    "base_lr": 5e-4,
    "warmup_steps": 2000, 
    "total_steps": (batches_per_epoch * 2) // accumulation_steps, # Total steps for 2 epochs
    "min_ratio": 0.1
}

# ==========================================
# 4. Execute Epoch 2 with Curriculum Masking
# ==========================================
print("\n=== RESUMING PHASE 1: Syntax (TinyStories) ===")
print("Curriculum Masking Active: Max Mask Capped at 40%")

current_epoch = 2

# Call the updated train_epoch function
avg_loss = train_epoch(
    model=model, 
    dataloader=tinystories_loader, 
    optimizer=optimizer, 
    scaler=scaler, 
    epoch=current_epoch, 
    phase_name="TinyStories_Curriculum",
    ema_helper=ema_helper,
    lr_schedule=lr_schedule
)

print(f"--> Epoch {current_epoch} Complete. Avg Loss: {avg_loss:.4f}\n")

# Save the new Phase 1 completion weights
torch.save(model.state_dict(), f"g_trm_tinystories_epoch_{current_epoch}_curriculum.pt")
print("Saved successful Curriculum checkpoint!")

Initializing Environment for Epoch 2...
Loading model weights from Epoch 1...



=== RESUMING PHASE 1: Syntax (TinyStories) ===
Curriculum Masking Active: Max Mask Capped at 40%
Epoch 2 [TinyStories_Curriculum] | Batch 0/7813 | Mask: 23% | Loss: 5.5541
Epoch 2 [TinyStories_Curriculum] | Batch 50/7813 | Mask: 34% | Loss: 5.5737
Epoch 2 [TinyStories_Curriculum] | Batch 100/7813 | Mask: 37% | Loss: 5.5570
Epoch 2 [TinyStories_Curriculum] | Batch 150/7813 | Mask: 26% | Loss: 5.6019
Epoch 2 [TinyStories_Curriculum] | Batch 200/7813 | Mask: 36% | Loss: 5.5990
Epoch 2 [TinyStories_Curriculum] | Batch 250/7813 | Mask: 17% | Loss: 5.4316
Epoch 2 [TinyStories_Curriculum] | Batch 300/7813 | Mask: 28% | Loss: 5.5266
Epoch 2 [TinyStories_Curriculum] | Batch 350/7813 | Mask: 35% | Loss: 5.5138
Epoch 2 [TinyStories_Curriculum] | Batch 400/7813 | Mask: 23% | Loss: 5.4993
Epoch 2 [TinyStories_Curriculum] | Batch 450/7813 | Mask: 35% | Loss: 5.5148
Epoch 2 [TinyStories_Curriculum] | Batch 500/7813 | Mask: 32% | Loss: 5.5425
Epoch 2 [TinyStories_Curriculum] | Batch 550/7813 | Mask: 

KeyboardInterrupt: 

In [16]:
import torch
import torch.optim as optim
from upload import TinyRecursiveReasoningModel_ACTV1
# Assuming tokenizer, dataloader, config_dict, and TinyRecursiveReasoningModel_ACTV1 are already in memory

print("Initializing Environment for Epoch 2...")
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# ==========================================
# 1. Load the Epoch 1 Checkpoint
# ==========================================
print("Loading model weights from Epoch 1...")
model = TinyRecursiveReasoningModel_ACTV1(config_dict)
model.load_state_dict(torch.load("g_trm_tinystories_epoch_1.pt", map_location=device))
model.to(device)

# ==========================================
# 2. Setup Optimizer, Scaler, and EMA
# ==========================================
# Re-initialize the optimizer
optimizer = optim.AdamW(model.parameters(), lr=5e-4, weight_decay=0.01)
scaler = torch.amp.GradScaler('cuda')

# (Optional) Re-initialize EMA if you are using a custom EMA helper class
# ema_helper = EMAHelper(mu=0.999) 
# ema_helper.register(model)
ema_helper = None # Replace with your EMA instance if active

# ==========================================
# 3. Resume Learning Rate Scheduler
# ==========================================
# We want to start at the step where Epoch 1 left off to avoid resetting the warmup
batches_per_epoch = len(tinystories_loader) # e.g., 15,625
accumulation_steps = 1 # Update if you are using grad accumulation

lr_schedule = {
    "step": batches_per_epoch // accumulation_steps, # FAST-FORWARD to the start of Epoch 2
    "base_lr": 5e-4,
    "warmup_steps": 2000, 
    "total_steps": (batches_per_epoch * 2) // accumulation_steps, # Total steps for 2 epochs
    "min_ratio": 0.1
}

# ==========================================
# 4. Execute Epoch 2 with Curriculum Masking
# ==========================================
print("\n=== RESUMING PHASE 1: Syntax (TinyStories) ===")
print("Curriculum Masking Active: Max Mask Capped at 40%")

current_epoch = 2

# Call the updated train_epoch function
avg_loss = train_epoch(
    model=model, 
    dataloader=tinystories_loader, 
    optimizer=optimizer, 
    scaler=scaler, 
    epoch=current_epoch, 
    phase_name="TinyStories_Curriculum",
    ema_helper=ema_helper,
    lr_schedule=lr_schedule
)

print(f"--> Epoch {current_epoch} Complete. Avg Loss: {avg_loss:.4f}\n")

# Save the new Phase 1 completion weights
torch.save(model.state_dict(), f"g_trm_tinystories_epoch_{current_epoch}_curriculum.pt")
print("Saved successful Curriculum checkpoint!")

Initializing Environment for Epoch 2...
Loading model weights from Epoch 1...

=== RESUMING PHASE 1: Syntax (TinyStories) ===
Curriculum Masking Active: Max Mask Capped at 40%


RuntimeError: CUDA error: out of memory
CUDA kernel errors might be asynchronously reported at some other API call, so the stacktrace below might be incorrect.
For debugging consider passing CUDA_LAUNCH_BLOCKING=1
Compile with `TORCH_USE_CUDA_DSA` to enable device-side assertions.
